<a href="https://colab.research.google.com/github/amit-eyal/apono-assignment/blob/main/sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import sqlite3

In [42]:
df = pd.read_csv('/content/JIT_Homework_Data.csv')
conn = sqlite3.connect('sql_analysis.db')
df.to_sql('customers', conn, if_exists= 'replace', index=False)
dups_query = 'SELECT CustomerID, COUNT(*) from customers GROUP BY CustomerID HAVING COUNT(*) > 1'
check_id_dups_df = pd.read_sql_query(dups_query, conn)
print(len(check_id_dups_df))

df.to_sql('customers', conn, if_exists= 'replace', index=False)
df.info()
df.describe(include='all')
df.head()

0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CustomerID                        100 non-null    object 
 1   AccountType                       100 non-null    object 
 2   RenewalProbability                100 non-null    float64
 3   SupportTickets                    100 non-null    int64  
 4   Policy EnforcementUsage           100 non-null    float64
 5   Policy EnforcementSatisfaction    100 non-null    int64  
 6   JIT Token GenerationUsage         100 non-null    float64
 7   JIT Token GenerationSatisfaction  100 non-null    int64  
 8   Access LogsUsage                  100 non-null    float64
 9   Access LogsSatisfaction           100 non-null    int64  
 10  Compliance ReportingUsage         100 non-null    float64
 11  Compliance ReportingSatisfaction  100 non-null    int64  
 12  Admin D

,CustomerID,AccountType,RenewalProbability,SupportTickets,Policy EnforcementUsage,Policy EnforcementSatisfaction,JIT Token GenerationUsage,JIT Token GenerationSatisfaction,Access LogsUsage,Access LogsSatisfaction,Compliance ReportingUsage,Compliance ReportingSatisfaction,Admin DashboardUsage,Admin DashboardSatisfaction,Access Delays,Compliance Questions,Integration Issues,Other
0,CUST1,Mid-Market,51.57,7,0.34,5,0.91,5,0.35,2,0.00,3,0.27,5,2,0,4,2
1,CUST2,Enterprise,81.82,3,0.38,2,0.11,3,0.40,1,0.33,1,0.59,3,4,4,4,4
2,CUST3,Enterprise,65.72,0,0.09,5,0.49,1,0.10,2,0.40,2,0.36,5,1,4,4,4
3,CUST4,Mid-Market,75.43,7,0.58,3,0.01,1,0.74,3,0.54,3,0.09,2,3,2,2,3
4,CUST5,Small Business,95.38,3,0.04,5,0.47,5,0.18,5,0.92,1,0.92,3,1,1,3,4


In [52]:
# 1) Feature Renewals
query = 'SELECT * FROM customers WHERE(customers.AccountType= "Enterprise")'
enterprise_df = pd.read_sql_query(query, conn)
list_columns = [c for c in enterprise_df.columns if 'Satisfaction' in c or 'Usage' in c]+ ['RenewalProbability']
# print(list_columns)
enterprise_df = enterprise_df[list_columns]
renewal_corr = enterprise_df.corr(numeric_only=True)
renewal_corr = renewal_corr['RenewalProbability'].sort_values(ascending=False)[1:]
# print(renewal_corr)
renewal_corr = renewal_corr.reset_index()
renewal_corr.columns = ['feature', 'corr']
renewal_corr['feature'] = renewal_corr['feature'].str.replace('Usage', '').str.replace('Satisfaction', '')
# print(renewal_corr)
linked_features = renewal_corr.groupby('feature')['corr'].mean().reset_index().sort_values(by='corr', ascending=False)
print(linked_features)


                feature      corr
0    Policy Enforcement  0.283821
1       Admin Dashboard  0.190963
2       Admin Dashboard  0.157179
3  Compliance Reporting  0.139997
4           Access Logs  0.117396
5    Policy Enforcement  0.104866
6  Compliance Reporting  0.000673
7  JIT Token Generation -0.004637
8           Access Logs -0.074273
9  JIT Token Generation -0.187034
                feature      corr
4    Policy Enforcement  0.194344
1       Admin Dashboard  0.174071
2  Compliance Reporting  0.070335
0           Access Logs  0.021561
3  JIT Token Generation -0.095836


In [86]:
# 2) Adoption Improvement

query = 'SELECT * FROM customers WHERE(customers.AccountType= "Mid-Market")'
mid_market_df = pd.read_sql_query(query, conn)
list_usage_columns = [c for c in mid_market_df.columns if 'Usage' in c]
mid_market_featurs_df = mid_market_df[list_usage_columns]
mid_market_featurs_df = mid_market_featurs_df.rename(columns=lambda x: 'avg_' + x.replace('Usage', '').replace(' ', '_'))
mid_market_featurs_df = mid_market_featurs_df.mean().sort_values(ascending=True)
print(mid_market_featurs_df)

tickets_query = 'SELECT AVG("Access Delays") AS avg_Access_Delays, AVG("Compliance Questions") AS aivg_Compliance_Questions, AVG("Integration Issues") AS avg_Integration_Issues, AVG(Other) as avg_Other\
 FROM customers WHERE customers.AccountType = "Mid-Market" GROUP BY AccountType'
mid_market_tickets_df = pd.read_sql_query(tickets_query, conn).iloc[0]
mid_market_tickets_df = mid_market_tickets_df.sort_values(ascending=False)
print(mid_market_tickets_df)




avg_Admin_Dashboard         0.435000
avg_Policy_Enforcement      0.453056
avg_Compliance_Reporting    0.502500
avg_Access_Logs             0.596944
avg_JIT_Token_Generation    0.636944
dtype: float64
avg_Integration_Issues       2.111111
avg_Access_Delays            2.083333
aivg_Compliance_Questions    2.000000
avg_Other                    1.861111
Name: 0, dtype: float64
